# Camada Silver

A camada Silver corresponde à etapa de tratamento, padronização e organização dos dados provenientes da camada Bronze.

Nesta fase, os arquivos consolidados anteriormente serão preparados para análise, mantendo a rastreabilidade das fontes e aplicando somente os ajustes necessários para tornar os dados consistentes e integráveis.

## Objetivos da camada Silver

Nesta etapa serão realizados:

- padronização dos nomes das colunas;
- ajuste dos tipos de dados;
- tratamento de valores ausentes quando necessário;
- reorganização de estruturas de cabeçalho;
- seleção das variáveis relevantes para o objetivo do projeto;
- preparação das bases para integração posterior na camada Gold.

Cada fonte será tratada separadamente antes de qualquer cruzamento entre os dados.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
pasta_bronze = Path("dados/bronze")
pasta_silver = Path("dados/silver")

In [3]:
arquivo_tab3_bronze = pasta_bronze / "ibge_grupos_etarios_csl.csv"

df_tab3 = pd.read_csv(arquivo_tab3_bronze)

df_tab3.head()

,ANO,CÓD.,SIGLA,LOCAL,POP_T,POP_H,POP_M,0-1_T,0-1_H,0-1_M,...,P_15-64_M,P_60+_T,P_60+_H,P_60+_M,P_65+_T,P_65+_H,P_65+_M,P_80+_T,P_80+_H,P_80+_M
0,2000,0,BR,Brasil,174695935,85720706,88975229,3423475,1739603,1683872,...,0.326749,0.087180,0.039357,0.047823,0.060261,0.026581,0.033680,0.011633,0.004666,0.006967
1,2001,0,BR,Brasil,177003743,86821427,90182316,3347313,1704785,1642528,...,0.329195,0.088406,0.039845,0.048562,0.061250,0.026957,0.034293,0.011851,0.004749,0.007102
2,2002,0,BR,Brasil,179228254,87882321,91345933,3274356,1671125,1603231,...,0.331573,0.089695,0.040361,0.049334,0.062338,0.027381,0.034957,0.012125,0.004845,0.007279
3,2003,0,BR,Brasil,181377654,88907299,92470355,3212295,1642237,1570058,...,0.333854,0.091074,0.040919,0.050156,0.063486,0.027834,0.035652,0.012443,0.004952,0.007491
4,2004,0,BR,Brasil,183469593,89905311,93564282,3163041,1619035,1544006,...,0.336031,0.092611,0.041550,0.051061,0.064668,0.028304,0.036364,0.012801,0.005070,0.007731


In [4]:
df_tab3.columns.tolist()

['ANO',
 'CÓD.',
 'SIGLA',
 'LOCAL',
 'POP_T',
 'POP_H',
 'POP_M',
 '0-1_T',
 '0-1_H',
 '0-1_M',
 '0-4_T',
 '0-4_H',
 '0-4_M',
 '0-14_T',
 '0-14_H',
 '0-14_M',
 '15-17_T',
 '15-17_H',
 '15-17_M',
 '18-21_T',
 '18-21_H',
 '18-21_M',
 '15-59_T',
 '15-59_H',
 '15-59_M',
 '15-64_T',
 '15-64_H',
 '15-64_M',
 '60+_T',
 '60+_H',
 '60+_M',
 '65+_T',
 '65+_H',
 '65+_M',
 '80+_T',
 '80+_H',
 '80+_M',
 'P_H',
 'P_M',
 'P_0-1_T',
 'P_0-1_H',
 'P_0-1_M',
 'P_0-4_T',
 'P_0-4_H',
 'P_0-4_M',
 'P_0-14_T',
 'P_0-14_H',
 'P_0-14_M',
 'P_15-17_T',
 'P_15-17_H',
 'P_15-17_M',
 'P_18-21_T',
 'P_18-21_H',
 'P_18-21_M',
 'P_15-59_T',
 'P_15-59_H',
 'P_15-59_M',
 'P_15-64_T',
 'P_15-64_H',
 'P_15-64_M',
 'P_60+_T',
 'P_60+_H',
 'P_60+_M',
 'P_65+_T',
 'P_65+_H',
 'P_65+_M',
 'P_80+_T',
 'P_80+_H',
 'P_80+_M']

### Seleção das variáveis demográficas

A base de grupos etários do IBGE possui 69 variáveis, incluindo população total, distribuição por sexo, diferentes grupos etários e suas respectivas proporções.

Como o projeto busca analisar o processo de envelhecimento populacional e sua relação com a população potencialmente ativa, serão selecionadas apenas as variáveis necessárias para responder às questões propostas.

Essa seleção reduz a complexidade da base sem alterar os dados originais preservados na camada Bronze.

In [5]:
pd.DataFrame({
    "indice": range(len(df_tab3.columns)),
    "coluna": df_tab3.columns
}).iloc[5:64]

,indice,coluna
5,5,POP_H
6,6,POP_M
7,7,0-1_T
8,8,0-1_H
9,9,0-1_M
10,10,0-4_T
11,11,0-4_H
12,12,0-4_M
13,13,0-14_T
14,14,0-14_H


### Definição das variáveis de interesse

A base contém diferentes recortes de idade, sexo e participação proporcional na população. Para manter a análise alinhada ao objetivo do projeto, serão priorizadas as variáveis relacionadas à transformação da estrutura etária.

O foco será dado à população jovem, à população em idade potencialmente ativa e aos grupos idosos, permitindo acompanhar o processo de envelhecimento populacional ao longo do tempo.

Antes da seleção definitiva das variáveis, serão verificadas as escalas utilizadas nos indicadores proporcionais disponibilizados pela fonte.

In [6]:
df_tab3.loc[
    (df_tab3["LOCAL"] == "Brasil") &
    (df_tab3["ANO"].isin([2000, 2024, 2050, 2070])),
    ["ANO", "POP_T", "0-14_T", "15-64_T", "60+_T", "65+_T",
     "P_0-14_T", "P_15-64_T", "P_60+_T", "P_65+_T"]
]

,ANO,POP_T,0-14_T,15-64_T,60+_T,65+_T,P_0-14_T,P_15-64_T,P_60+_T,P_65+_T
0,2000,174695935,52259915,111908697,15229921,10527323,0.299148,0.640591,0.087180,0.060261
24,2024,212583750,42054707,146815459,34169617,23713584,0.197827,0.690624,0.160735,0.111549
50,2050,218369418,29743299,137950440,65549776,50675679,0.136206,0.631730,0.300178,0.232064
70,2070,199228708,23808343,113603532,75292150,61816833,0.119503,0.570217,0.377918,0.310281


In [7]:
colunas_tab3 = [
    "ANO",
    "CÓD.",
    "SIGLA",
    "LOCAL",
    "POP_T",
    "0-14_T",
    "15-64_T",
    "60+_T",
    "65+_T",
    "80+_T",
    "P_0-14_T",
    "P_15-64_T",
    "P_60+_T",
    "P_65+_T",
    "P_80+_T"
]

df_tab3_silver = df_tab3[colunas_tab3].copy()

df_tab3_silver.head()

,ANO,CÓD.,SIGLA,LOCAL,POP_T,0-14_T,15-64_T,60+_T,65+_T,80+_T,P_0-14_T,P_15-64_T,P_60+_T,P_65+_T,P_80+_T
0,2000,0,BR,Brasil,174695935,52259915,111908697,15229921,10527323,2032255,0.299148,0.640591,0.087180,0.060261,0.011633
1,2001,0,BR,Brasil,177003743,51985452,114176772,15648261,10841519,2097640,0.293697,0.645053,0.088406,0.061250,0.011851
2,2002,0,BR,Brasil,179228254,51668727,116386875,16075850,11172652,2173094,0.288284,0.649378,0.089695,0.062338,0.012125
3,2003,0,BR,Brasil,181377654,51329911,118532883,16518858,11514860,2256815,0.283000,0.653514,0.091074,0.063486,0.012443
4,2004,0,BR,Brasil,183469593,50980818,120624088,16991330,11864687,2348541,0.277871,0.657461,0.092611,0.064668,0.012801


In [8]:
df_tab3_silver.shape

(2343, 15)

### Padronização dos nomes das variáveis

Após a seleção das variáveis de interesse, os nomes das colunas serão padronizados para facilitar sua identificação e utilização nas próximas etapas do processamento.

As nomenclaturas originais do IBGE serão substituídas por nomes mais descritivos, mantendo o significado das variáveis e diferenciando valores absolutos de proporções populacionais.

In [9]:
df_tab3_silver = df_tab3_silver.rename(columns={
    "ANO": "ano",
    "CÓD.": "codigo",
    "SIGLA": "sigla",
    "LOCAL": "local",
    "POP_T": "populacao_total",
    "0-14_T": "populacao_0_14",
    "15-64_T": "populacao_15_64",
    "60+_T": "populacao_60_mais",
    "65+_T": "populacao_65_mais",
    "80+_T": "populacao_80_mais",
    "P_0-14_T": "proporcao_0_14",
    "P_15-64_T": "proporcao_15_64",
    "P_60+_T": "proporcao_60_mais",
    "P_65+_T": "proporcao_65_mais",
    "P_80+_T": "proporcao_80_mais"
})

df_tab3_silver.head()

,ano,codigo,sigla,local,populacao_total,populacao_0_14,populacao_15_64,populacao_60_mais,populacao_65_mais,populacao_80_mais,proporcao_0_14,proporcao_15_64,proporcao_60_mais,proporcao_65_mais,proporcao_80_mais
0,2000,0,BR,Brasil,174695935,52259915,111908697,15229921,10527323,2032255,0.299148,0.640591,0.087180,0.060261,0.011633
1,2001,0,BR,Brasil,177003743,51985452,114176772,15648261,10841519,2097640,0.293697,0.645053,0.088406,0.061250,0.011851
2,2002,0,BR,Brasil,179228254,51668727,116386875,16075850,11172652,2173094,0.288284,0.649378,0.089695,0.062338,0.012125
3,2003,0,BR,Brasil,181377654,51329911,118532883,16518858,11514860,2256815,0.283000,0.653514,0.091074,0.063486,0.012443
4,2004,0,BR,Brasil,183469593,50980818,120624088,16991330,11864687,2348541,0.277871,0.657461,0.092611,0.064668,0.012801


### Validação da base tratada

Após a seleção e padronização das variáveis, será realizada uma verificação da consistência da base tratada.

Nesta etapa serão avaliados os tipos de dados, a presença de valores ausentes e a existência de registros duplicados, garantindo que a base esteja adequada antes de sua consolidação na camada Silver.

In [10]:
df_tab3_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2343 entries, 0 to 2342
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ano                2343 non-null   int64  
 1   codigo             2343 non-null   int64  
 2   sigla              2343 non-null   object 
 3   local              2343 non-null   object 
 4   populacao_total    2343 non-null   int64  
 5   populacao_0_14     2343 non-null   int64  
 6   populacao_15_64    2343 non-null   int64  
 7   populacao_60_mais  2343 non-null   int64  
 8   populacao_65_mais  2343 non-null   int64  
 9   populacao_80_mais  2343 non-null   int64  
 10  proporcao_0_14     2343 non-null   float64
 11  proporcao_15_64    2343 non-null   float64
 12  proporcao_60_mais  2343 non-null   float64
 13  proporcao_65_mais  2343 non-null   float64
 14  proporcao_80_mais  2343 non-null   float64
dtypes: float64(5), int64(8), object(2)
memory usage: 274.7+ KB


In [11]:
df_tab3_silver.isnull().sum()

ano                  0
codigo               0
sigla                0
local                0
populacao_total      0
populacao_0_14       0
populacao_15_64      0
populacao_60_mais    0
populacao_65_mais    0
populacao_80_mais    0
proporcao_0_14       0
proporcao_15_64      0
proporcao_60_mais    0
proporcao_65_mais    0
proporcao_80_mais    0
dtype: int64

In [12]:
df_tab3_silver.duplicated().sum()

np.int64(0)

In [13]:
df_tab3_silver.duplicated(
    subset=["ano", "codigo", "sigla", "local"]
).sum()

np.int64(0)

### Resultado do tratamento da base de grupos etários

Após o processo de seleção e padronização, a base de grupos etários foi reduzida de 69 para 15 variáveis relevantes ao objetivo do projeto, mantendo seus 2.343 registros.

A validação não identificou valores ausentes, registros duplicados ou duplicidades na combinação de ano e localidade. Os tipos de dados também se mostraram adequados às variáveis analisadas.

Com essas verificações concluídas, a base está preparada para armazenamento na camada Silver.

In [14]:
arquivo_silver_tab3 = pasta_silver / "ibge_grupos_etarios_tratado.csv"

df_tab3_silver.to_csv(
    arquivo_silver_tab3,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo salvo em: {arquivo_silver_tab3}")

Arquivo salvo em: dados\silver\ibge_grupos_etarios_tratado.csv


In [15]:
arquivo_silver_tab3.exists()

True

### Tratamento da base de indicadores demográficos

Com a base de grupos etários concluída na camada Silver, inicia-se o tratamento da base de indicadores demográficos do IBGE.

Nesta etapa, serão selecionadas apenas as variáveis relacionadas ao objetivo do projeto, mantendo os indicadores necessários para analisar fecundidade, longevidade, envelhecimento populacional e dependência demográfica.

In [16]:
arquivo_tab4_bronze = pasta_bronze / "ibge_indicadores_csl.csv"

df_tab4 = pd.read_csv(arquivo_tab4_bronze)

df_tab4.head()

,ANO,CÓD.,SIGLA,LOCAL,POP_T,POP_H,POP_M,TCG_T,TCG_H,TCG_M,...,RDI60,RDT60,RDJ65,RDI65,RDT65,IE60,IE65,IE80,ID_M,ID_MED
0,2000,0,BR,Brasil,174695935,85720706,88975229,NaN,NaN,NaN,...,14.206208,62.953355,46.698707,9.407064,56.105772,29.142644,20.144164,3.888745,28.314517,25.286941
1,2001,0,BR,Brasil,177003743,86821427,90182316,1.321043,1.284078,1.356655,...,14.307632,61.839348,45.530672,9.495381,55.026053,30.101231,20.854910,4.035052,28.572942,25.595223
2,2002,0,BR,Brasil,179228254,87882321,91345933,1.256759,1.221926,1.290294,...,14.419914,60.766364,44.393946,9.599581,53.993527,31.113308,21.623625,4.205821,28.845196,25.923454
3,2003,0,BR,Brasil,181377654,88907299,92470355,1.199253,1.166307,1.230949,...,14.550357,59.763442,43.304364,9.714486,53.018850,32.181739,22.433041,4.396686,29.128255,26.278205
4,2004,0,BR,Brasil,183469593,89905311,93564282,1.153361,1.122531,1.183003,...,14.711434,58.851646,42.264210,9.836084,52.100294,33.328869,23.272845,4.606715,29.420245,26.650273


In [17]:
pd.DataFrame({
    "indice": range(len(df_tab4.columns)),
    "coluna": df_tab4.columns
})

,indice,coluna
0,0,ANO
1,1,CÓD.
2,2,SIGLA
3,3,LOCAL
4,4,POP_T
5,5,POP_H
6,6,POP_M
7,7,TCG_T
8,8,TCG_H
9,9,TCG_M


### Definição das variáveis de interesse

A base de indicadores demográficos contém 59 variáveis relacionadas à dinâmica populacional, mortalidade, fecundidade, longevidade, dependência e envelhecimento.

Para a camada Silver, serão priorizados os indicadores diretamente relacionados ao objetivo do projeto: acompanhar a transformação da estrutura demográfica brasileira e fornecer elementos para sua posterior comparação com os dados de contribuintes da Previdência Social.

Antes da seleção definitiva, os principais indicadores serão observados em anos estratégicos da série histórica.

In [18]:
df_tab4.loc[
    (df_tab4["LOCAL"] == "Brasil") &
    (df_tab4["ANO"].isin([2000, 2024, 2050, 2070])),
    [
        "ANO", "POP_T", "TCG_T",
        "e0_T", "e60_T", "TFT",
        "RDJ60", "RDI60", "RDT60",
        "IE60", "ID_M", "ID_MED"
    ]
]

,ANO,POP_T,TCG_T,e0_T,e60_T,TFT,RDJ60,RDI60,RDT60,IE60,ID_M,ID_MED
0,2000,174695935,NaN,71.102185,20.109002,2.315552,48.747147,14.206208,62.953355,29.142644,28.314517,25.286941
24,2024,212583750,0.419751,76.605711,22.590940,1.546803,30.841071,25.058493,55.899564,81.250398,35.850492,35.251142
50,2050,218369418,-0.202002,81.322136,25.145951,1.450256,24.166544,53.259444,77.425988,220.385022,44.285912,45.687653
70,2070,199228708,-0.670412,83.910428,26.532279,1.495605,23.777856,75.195738,98.973594,316.242714,48.434501,51.168685


### Seleção dos indicadores demográficos

A análise dos anos de 2000, 2024, 2050 e 2070 evidenciou mudanças relevantes na estrutura demográfica brasileira, como a redução da fecundidade e do crescimento populacional, o aumento da longevidade e a elevação dos indicadores de envelhecimento e dependência da população idosa.

Com base nessa avaliação, foram selecionadas as variáveis diretamente relacionadas ao objetivo do projeto, reduzindo a base original de 59 colunas para um conjunto de indicadores demográficos de interesse.

In [19]:
colunas_tab4 = [
    "ANO", "CÓD.", "SIGLA", "LOCAL",
    "POP_T", "TCG_T",
    "e0_T", "e60_T", "TFT",
    "RDJ60", "RDI60", "RDT60",
    "IE60", "ID_M", "ID_MED"
]

df_tab4_silver = df_tab4[colunas_tab4].copy()

df_tab4_silver.head()

,ANO,CÓD.,SIGLA,LOCAL,POP_T,TCG_T,e0_T,e60_T,TFT,RDJ60,RDI60,RDT60,IE60,ID_M,ID_MED
0,2000,0,BR,Brasil,174695935,NaN,71.102185,20.109002,2.315552,48.747147,14.206208,62.953355,29.142644,28.314517,25.286941
1,2001,0,BR,Brasil,177003743,1.321043,71.502797,20.255832,2.149242,47.531716,14.307632,61.839348,30.101231,28.572942,25.595223
2,2002,0,BR,Brasil,179228254,1.256759,71.824406,20.361073,2.066614,46.346450,14.419914,60.766364,31.113308,28.845196,25.923454
3,2003,0,BR,Brasil,181377654,1.199253,72.103568,20.440327,2.016345,45.213085,14.550357,59.763442,32.181739,29.128255,26.278205
4,2004,0,BR,Brasil,183469593,1.153361,72.587444,20.629628,1.969960,44.140213,14.711434,58.851646,33.328869,29.420245,26.650273


### Padronização dos nomes das variáveis

Após a seleção das variáveis de interesse, os nomes das colunas serão padronizados para facilitar a leitura, manipulação e integração dos dados nas próximas etapas do projeto.

A padronização altera apenas a nomenclatura das variáveis, mantendo os dados originais preservados.

In [20]:
df_tab4_silver = df_tab4_silver.rename(columns={
    "ANO": "ano",
    "CÓD.": "codigo",
    "SIGLA": "sigla",
    "LOCAL": "local",
    "POP_T": "populacao_total",
    "TCG_T": "taxa_crescimento",
    "e0_T": "expectativa_vida",
    "e60_T": "expectativa_vida_60",
    "TFT": "taxa_fecundidade",
    "RDJ60": "razao_dependencia_jovens",
    "RDI60": "razao_dependencia_idosos",
    "RDT60": "razao_dependencia_total",
    "IE60": "indice_envelhecimento",
    "ID_M": "idade_media",
    "ID_MED": "idade_mediana"
})

df_tab4_silver.head()

,ano,codigo,sigla,local,populacao_total,taxa_crescimento,expectativa_vida,expectativa_vida_60,taxa_fecundidade,razao_dependencia_jovens,razao_dependencia_idosos,razao_dependencia_total,indice_envelhecimento,idade_media,idade_mediana
0,2000,0,BR,Brasil,174695935,NaN,71.102185,20.109002,2.315552,48.747147,14.206208,62.953355,29.142644,28.314517,25.286941
1,2001,0,BR,Brasil,177003743,1.321043,71.502797,20.255832,2.149242,47.531716,14.307632,61.839348,30.101231,28.572942,25.595223
2,2002,0,BR,Brasil,179228254,1.256759,71.824406,20.361073,2.066614,46.346450,14.419914,60.766364,31.113308,28.845196,25.923454
3,2003,0,BR,Brasil,181377654,1.199253,72.103568,20.440327,2.016345,45.213085,14.550357,59.763442,32.181739,29.128255,26.278205
4,2004,0,BR,Brasil,183469593,1.153361,72.587444,20.629628,1.969960,44.140213,14.711434,58.851646,33.328869,29.420245,26.650273


### Validação da estrutura da base tratada

Com as variáveis selecionadas e seus nomes padronizados, será realizada a validação da estrutura da base.

Nesta etapa, serão verificados os tipos de dados e a presença de valores ausentes, permitindo identificar possíveis necessidades de tratamento antes da consolidação definitiva na camada Silver.

In [21]:
df_tab4_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2343 entries, 0 to 2342
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ano                       2343 non-null   int64  
 1   codigo                    2343 non-null   int64  
 2   sigla                     2343 non-null   object 
 3   local                     2343 non-null   object 
 4   populacao_total           2343 non-null   int64  
 5   taxa_crescimento          2310 non-null   float64
 6   expectativa_vida          2343 non-null   float64
 7   expectativa_vida_60       2343 non-null   float64
 8   taxa_fecundidade          2343 non-null   float64
 9   razao_dependencia_jovens  2343 non-null   float64
 10  razao_dependencia_idosos  2343 non-null   float64
 11  razao_dependencia_total   2343 non-null   float64
 12  indice_envelhecimento     2343 non-null   float64
 13  idade_media               2343 non-null   float64
 14  idade_me

In [22]:
df_tab4_silver.isnull().sum()

ano                          0
codigo                       0
sigla                        0
local                        0
populacao_total              0
taxa_crescimento            33
expectativa_vida             0
expectativa_vida_60          0
taxa_fecundidade             0
razao_dependencia_jovens     0
razao_dependencia_idosos     0
razao_dependencia_total      0
indice_envelhecimento        0
idade_media                  0
idade_mediana                0
dtype: int64

### Verificação dos valores ausentes na taxa de crescimento

A validação identificou 33 valores ausentes exclusivamente na variável `taxa_crescimento`.

Como a série histórica tem início no ano 2000 e a taxa de crescimento depende da comparação com um período anterior, será verificado se essas ausências correspondem ao primeiro ano disponível para cada localidade.

Caso esse padrão seja confirmado, os valores ausentes serão preservados por representarem uma característica metodológica da própria série, e não uma falha de preenchimento dos dados.

In [23]:
df_tab4_silver.loc[
    df_tab4_silver["taxa_crescimento"].isnull(),
    ["ano", "codigo", "sigla", "local", "taxa_crescimento"]
]

,ano,codigo,sigla,local,taxa_crescimento
0,2000,0,BR,Brasil,NaN
71,2000,1,NO,Norte,NaN
142,2000,2,ND,Nordeste,NaN
213,2000,3,SD,Sudeste,NaN
284,2000,4,SU,Sul,NaN
355,2000,5,CO,Centro-Oeste,NaN
426,2000,11,RO,Rondônia,NaN
497,2000,12,AC,Acre,NaN
568,2000,13,AM,Amazonas,NaN
639,2000,14,RR,Roraima,NaN


### Tratamento dos valores ausentes

Os 33 valores ausentes identificados na variável `taxa_crescimento` correspondem exclusivamente ao ano de 2000, primeiro período disponível para cada unidade territorial da série.

Como o cálculo da taxa de crescimento depende de um período anterior, essas ausências são compatíveis com a estrutura temporal do indicador e não representam falhas de preenchimento.

Dessa forma, os valores serão preservados na camada Silver, sem aplicação de preenchimento ou exclusão de registros.

In [24]:
df_tab4_silver.duplicated().sum()

np.int64(0)

In [25]:
df_tab4_silver.duplicated(
    subset=["ano", "codigo", "sigla", "local"]
).sum()

np.int64(0)

### Resultado do tratamento da base de indicadores demográficos

Após a seleção e padronização das variáveis, a base de indicadores demográficos foi reduzida de 59 para 15 colunas, mantendo seus 2.343 registros.

A validação não identificou registros duplicados nem duplicidades na combinação de ano e localidade. Os 33 valores ausentes encontrados na taxa de crescimento foram mantidos por corresponderem ao primeiro ano da série de cada unidade territorial.

Com essas verificações concluídas, a base está preparada para armazenamento na camada Silver.

In [26]:
arquivo_silver_tab4 = pasta_silver / "ibge_indicadores_tratado.csv"

df_tab4_silver.to_csv(
    arquivo_silver_tab4,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo salvo em: {arquivo_silver_tab4}")

Arquivo salvo em: dados\silver\ibge_indicadores_tratado.csv


In [27]:
arquivo_silver_tab4.exists()

True

### Tratamento da base de contribuintes da Previdência Social

Após a conclusão do tratamento das duas bases demográficas do IBGE, inicia-se a preparação da base de contribuintes da Previdência Social.

Diferentemente das bases anteriores, esta fonte apresenta uma estrutura tabular própria, com informações de faixa etária, ano, total de contribuintes e distribuição por sexo.

Nesta etapa inicial, a estrutura preservada na camada Bronze será novamente inspecionada para definir a organização adequada dos dados na camada Silver.

In [28]:
arquivo_aeps_bronze = pasta_bronze / "aeps_contribuintes_csl.csv"

df_aeps = pd.read_csv(
    arquivo_aeps_bronze,
    header=None
)

df_aeps

,0,1,2,3,4,5
0,32.3 - Número médio mensal de contribuintes pe...,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,/ ANOS,NaN,NÚMERO MÉDIO MENSAL DE CONTRIBUINTES PESSOAS F...,NaN,NaN,NaN
3,NaN,NaN,Total,Sexo,NaN,NaN
4,NaN,NaN,NaN,Masculino,Feminino,Ignorado
5,TOTAL,2022.0,58272885,31236748,26960253,75884
6,NaN,2023.0,60532508,32397067,28112041,23400
7,NaN,2024.0,62180770,33230738,28926291,23740
8,Até 19 anos,2022.0,1279843,683859,589057,6927
9,NaN,2023.0,1379548,735533,643945,70


### Identificação da estrutura da base de contribuintes

A inspeção da base do AEPS mostrou que as cinco primeiras linhas correspondem à estrutura de cabeçalho da tabela, enquanto os registros de contribuintes começam a partir da linha seguinte.

Os dados estão organizados por faixa etária e ano, contendo o número total de contribuintes e sua distribuição entre masculino, feminino e sexo ignorado.

Antes da transformação da estrutura, será verificado o limite final dos registros para separar os dados das notas e informações complementares existentes na fonte original.

In [29]:
df_aeps.tail(15)

,0,1,2,3,4,5
32,55 a 59 anos,2022.0,4186975,2176826,2008271,1878
33,NaN,2023.0,4419284,2302465,2116136,683
34,NaN,2024.0,4556774,2377535,2178552,687
35,60 a 64 anos,2022.0,2428106,1347149,1080087,871
36,NaN,2023.0,2683598,1474650,1208655,292
37,NaN,2024.0,2882947,1585305,1297285,357
38,65 a 69 anos,2022.0,872981,509756,362935,290
39,NaN,2023.0,950282,559611,390524,148
40,NaN,2024.0,1018073,604347,413579,147
41,70 anos e mais,2022.0,446531,272443,173930,157


### Estruturação dos registros de contribuintes

A inspeção confirmou que os registros de contribuintes estão compreendidos entre as linhas 5 e 46 da base, totalizando 42 observações.

As linhas de cabeçalho serão removidas e as seis variáveis serão identificadas como faixa etária, ano, total de contribuintes, masculino, feminino e sexo ignorado.

Neste primeiro momento, os valores ausentes existentes na identificação das faixas etárias serão preservados para posterior tratamento.

In [30]:
df_aeps_silver = df_aeps.iloc[5:47].copy()

df_aeps_silver.columns = [
    "faixa_etaria",
    "ano",
    "total_contribuintes",
    "masculino",
    "feminino",
    "ignorado"
]

df_aeps_silver.head(10)

,faixa_etaria,ano,total_contribuintes,masculino,feminino,ignorado
5,TOTAL,2022.0,58272885,31236748,26960253,75884
6,NaN,2023.0,60532508,32397067,28112041,23400
7,NaN,2024.0,62180770,33230738,28926291,23740
8,Até 19 anos,2022.0,1279843,683859,589057,6927
9,NaN,2023.0,1379548,735533,643945,70
10,NaN,2024.0,1530144,809522,720578,43
11,20 a 24 anos,2022.0,5884867,3223538,2653205,8124
12,NaN,2023.0,6026451,3298627,2727517,307
13,NaN,2024.0,6127670,3343110,2784337,223
14,25 a 29 anos,2022.0,7712262,4147165,3559290,5808


### Preenchimento das faixas etárias

Na fonte original, cada faixa etária é apresentada apenas no primeiro registro do grupo, enquanto os anos seguintes permanecem com a célula vazia.

Para tornar cada registro independente e adequado à estrutura tabular da camada Silver, a identificação da faixa etária será propagada para os registros subsequentes do mesmo grupo.

In [31]:
df_aeps_silver["faixa_etaria"] = df_aeps_silver["faixa_etaria"].ffill()

df_aeps_silver.head(10)

,faixa_etaria,ano,total_contribuintes,masculino,feminino,ignorado
5,TOTAL,2022.0,58272885,31236748,26960253,75884
6,TOTAL,2023.0,60532508,32397067,28112041,23400
7,TOTAL,2024.0,62180770,33230738,28926291,23740
8,Até 19 anos,2022.0,1279843,683859,589057,6927
9,Até 19 anos,2023.0,1379548,735533,643945,70
10,Até 19 anos,2024.0,1530144,809522,720578,43
11,20 a 24 anos,2022.0,5884867,3223538,2653205,8124
12,20 a 24 anos,2023.0,6026451,3298627,2727517,307
13,20 a 24 anos,2024.0,6127670,3343110,2784337,223
14,25 a 29 anos,2022.0,7712262,4147165,3559290,5808


### Verificação das faixas etárias

Após o preenchimento dos valores ausentes, cada registro passou a possuir sua respectiva identificação de faixa etária.

A próxima verificação consiste em identificar todas as categorias presentes na base, permitindo compreender sua organização antes das etapas finais de tratamento.

In [32]:
df_aeps_silver["faixa_etaria"].unique()

array(['TOTAL', 'Até 19 anos', '20 a 24 anos', '25 a 29 anos',
       '30 a 34 anos', '35 a 39 anos', '40 a 44 anos', '45 a 49 anos',
       '50 a 54 anos', '55 a 59 anos', '60 a 64 anos', '65 a 69 anos',
       '70 anos e mais', 'Ignorada'], dtype=object)

### Verificação dos tipos de dados

A base apresenta as faixas etárias esperadas, além das categorias `TOTAL` e `Ignorada`, que serão preservadas por representarem informações válidas da fonte original.

Com a estrutura das categorias confirmada, será realizada a inspeção dos tipos de dados antes de qualquer conversão necessária para a camada Silver.

In [33]:
df_aeps_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 5 to 46
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   faixa_etaria         42 non-null     object 
 1   ano                  42 non-null     float64
 2   total_contribuintes  42 non-null     object 
 3   masculino            42 non-null     object 
 4   feminino             42 non-null     object 
 5   ignorado             42 non-null     object 
dtypes: float64(1), object(5)
memory usage: 2.1+ KB


### Padronização dos tipos de dados

A inspeção identificou que o ano foi interpretado como número decimal e que as variáveis quantitativas de contribuintes foram armazenadas como texto devido à estrutura original da fonte.

Para adequar a base à camada Silver, o ano será convertido para número inteiro e as variáveis de quantidade de contribuintes serão convertidas para tipos numéricos, mantendo seus valores originais.

In [34]:
df_aeps_silver["ano"] = df_aeps_silver["ano"].astype(int)

colunas_numericas = [
    "total_contribuintes",
    "masculino",
    "feminino",
    "ignorado"
]

df_aeps_silver[colunas_numericas] = (
    df_aeps_silver[colunas_numericas]
    .apply(pd.to_numeric)
    .astype(int)
)

df_aeps_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 5 to 46
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   faixa_etaria         42 non-null     object
 1   ano                  42 non-null     int64 
 2   total_contribuintes  42 non-null     int64 
 3   masculino            42 non-null     int64 
 4   feminino             42 non-null     int64 
 5   ignorado             42 non-null     int64 
dtypes: int64(5), object(1)
memory usage: 2.1+ KB


### Validação da consistência dos totais

Com os tipos de dados padronizados, será realizada uma verificação de consistência entre as variáveis quantitativas da base.

O objetivo é confirmar se o valor de `total_contribuintes` corresponde à soma das categorias `masculino`, `feminino` e `ignorado` em cada registro.

In [35]:
df_aeps_silver["soma_sexo"] = (
    df_aeps_silver["masculino"]
    + df_aeps_silver["feminino"]
    + df_aeps_silver["ignorado"]
)

df_aeps_silver["diferenca_total"] = (
    df_aeps_silver["total_contribuintes"]
    - df_aeps_silver["soma_sexo"]
)

df_aeps_silver[
    ["faixa_etaria", "ano", "total_contribuintes", "soma_sexo", "diferenca_total"]
]

,faixa_etaria,ano,total_contribuintes,soma_sexo,diferenca_total
5,TOTAL,2022,58272885,58272885,0
6,TOTAL,2023,60532508,60532508,0
7,TOTAL,2024,62180770,62180769,1
8,Até 19 anos,2022,1279843,1279843,0
9,Até 19 anos,2023,1379548,1379548,0
10,Até 19 anos,2024,1530144,1530143,1
11,20 a 24 anos,2022,5884867,5884867,0
12,20 a 24 anos,2023,6026451,6026451,0
13,20 a 24 anos,2024,6127670,6127670,0
14,25 a 29 anos,2022,7712262,7712263,-1


### Análise das diferenças entre os totais

A comparação entre o total de contribuintes informado na fonte e a soma das categorias por sexo revelou pequenas diferenças em alguns registros.

Para avaliar a extensão dessas diferenças antes de qualquer decisão de tratamento, serão identificados os registros em que o total informado não corresponde exatamente à soma das categorias.

In [36]:
df_aeps_silver[
    df_aeps_silver["diferenca_total"] != 0
][
    ["faixa_etaria", "ano", "total_contribuintes",
     "soma_sexo", "diferenca_total"]
]

,faixa_etaria,ano,total_contribuintes,soma_sexo,diferenca_total
7,TOTAL,2024,62180770,62180769,1
10,Até 19 anos,2024,1530144,1530143,1
14,25 a 29 anos,2022,7712262,7712263,-1
19,30 a 34 anos,2024,8027384,8027383,1
21,35 a 39 anos,2023,8282050,8282049,1
25,40 a 44 anos,2024,8301650,8301649,1
26,45 a 49 anos,2022,6323750,6323751,-1
28,45 a 49 anos,2024,7086247,7086248,-1
31,50 a 54 anos,2024,5699018,5699019,-1
35,60 a 64 anos,2022,2428106,2428107,-1


### Resultado da validação dos totais

A comparação entre o total de contribuintes informado na fonte e a soma das categorias Masculino, Feminino e Ignorado identificou pequenas divergências em 14 dos 42 registros analisados.

Em todos os casos divergentes, a diferença observada foi de apenas uma unidade (`+1` ou `-1`). Como os valores são provenientes da fonte original, essas diferenças serão preservadas, evitando alterações nos dados oficiais durante o tratamento.

A validação permite registrar essa característica da base e manter a rastreabilidade dos dados na camada Silver.

### Remoção das colunas auxiliares de validação

As colunas `soma_sexo` e `diferenca_total` foram criadas exclusivamente para verificar a consistência entre o total de contribuintes informado na fonte e a soma das categorias por sexo.

Como essas variáveis não pertencem à estrutura original da base nem são necessárias para as análises posteriores, elas serão removidas antes do armazenamento dos dados tratados na camada Silver.

In [37]:
df_aeps_silver = df_aeps_silver.drop(
    columns=["soma_sexo", "diferenca_total"]
)

df_aeps_silver.head()

,faixa_etaria,ano,total_contribuintes,masculino,feminino,ignorado
5,TOTAL,2022,58272885,31236748,26960253,75884
6,TOTAL,2023,60532508,32397067,28112041,23400
7,TOTAL,2024,62180770,33230738,28926291,23740
8,Até 19 anos,2022,1279843,683859,589057,6927
9,Até 19 anos,2023,1379548,735533,643945,70


### Verificação de registros duplicados

Com a estrutura final da base definida, será verificada a existência de registros duplicados.

Como cada combinação de faixa etária e ano deve representar uma única observação, a análise será realizada considerando essas duas variáveis como identificadoras dos registros.

In [38]:
df_aeps_silver.duplicated().sum()

np.int64(0)

In [39]:
df_aeps_silver.duplicated(
    subset=["faixa_etaria", "ano"]
).sum()

np.int64(0)

### Resultado da verificação de duplicidades

Não foram identificados registros duplicados na base, tanto na comparação das linhas completas quanto na combinação entre `faixa_etaria` e `ano`.

Dessa forma, cada faixa etária possui um único registro para cada ano analisado. Com as verificações concluídas, a base está preparada para armazenamento na camada Silver.

In [40]:
arquivo_silver_aeps = pasta_silver / "aeps_contribuintes_tratado.csv"

df_aeps_silver.to_csv(
    arquivo_silver_aeps,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo salvo em: {arquivo_silver_aeps}")

Arquivo salvo em: dados\silver\aeps_contribuintes_tratado.csv


In [41]:
arquivo_silver_aeps.exists()

True

## Tratamento da série histórica de contribuintes da Previdência Social

Após a incorporação da série histórica de contribuintes à camada Bronze, inicia-se o tratamento da base com informações anuais sobre a quantidade de contribuintes pessoas físicas no período de 2003 a 2023.

A inclusão desta série amplia o horizonte temporal da análise previdenciária, permitindo posteriormente comparar a evolução do número de contribuintes com as transformações demográficas observadas nas bases do IBGE.

Nesta etapa serão identificadas as variáveis relevantes, realizada a padronização da estrutura dos dados e verificadas possíveis inconsistências antes do armazenamento da base tratada na camada Silver.

In [44]:
arquivo_aeps_historico_contrib = (
    pasta_bronze / "aeps_historico_contribuintes.csv"
)

df_aeps_historico_contrib = pd.read_csv(
    arquivo_aeps_historico_contrib,
    header=None
)

df_aeps_historico_contrib.head(15)

,0,1,2,3,4,5,6,7,8,9,10
0,Capítulo 5 - Contribuintes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5.1 - Quantidade de contribuintes pessoas físi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANOS,QUANTIDADE DE CONTRIBUINTES PESSOAS FÍSICAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Total,Trabalhadores\nEmpregados (I),Outros Contribuintes (1),NaN,NaN,NaN,Ambos,NaN,NaN,NaN
5,NaN,NaN,NaN,Total,GPS (II),GFIP (III),GPS e GFIP\n(IV),Total,(I) e (II)\nconjuntamente,(I) e (III)\nconjuntamente,(I) e (IV)\nconjuntamente
6,2003,39850452,30537383,8395885,4704900,2120751,1570234,917184,254569,580788,81827
7,2004,42084323,32200411,8766902,4592168,3850082,324652,1117010,256764,835901,24345
8,2005,45035035,34687001,9099704,4693028,4114667,292009,1248330,280717,943844,23769
9,2006,46676737,36158570,9262079,4725699,4275536,260844,1256088,277327,958280,20481


In [45]:
df_aeps_historico_contrib.shape

(30, 11)

In [46]:
df_aeps_historico_contrib.tail(10)

,0,1,2,3,4,5,6,7,8,9,10
20,2017,65232942,50218289,13247745,8094774,4840740,312231,1766908,702721,1025719,38468
21,2018,65549513,49766448,13410055,8304268,4790530,315257,2373010,1052824,1266031,54155
22,2019,67092219,50262554,14140095,9062325,4791277,286493,2689570,1291437,1343249,54884
23,2020,65576866,48518045,14415371,9485371,4688702,241298,2643450,1357551,1239950,45949
24,2021,69447394,50362250,15658472,10327257,5071252,259963,3426672,1749076,1617719,59877
25,2022,72735346,52836383,16119440,10957713,5118285,43442,3779523,2031265,1734081,14177
26,2023,73982758,53709072,16520583,11151669,5368859,55,3753103,2024540,1728538,25
27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,"FONTE: DATAPREV, CNIS, Tabulação Especial GFIP.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,NOTA: As diferenças porventura existentes entr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Seleção dos registros históricos válidos

A inspeção da estrutura da base identificou que os dados históricos de contribuintes estão compreendidos entre os anos de **2003 e 2023**, totalizando **21 observações anuais consecutivas**.

As linhas anteriores aos registros correspondem aos cabeçalhos e informações de organização da tabela original do AEPS, enquanto as linhas posteriores contêm informações de fonte e notas metodológicas.

Nesta etapa serão mantidos apenas os registros anuais válidos. As colunas originais serão inicialmente preservadas para que suas informações possam ser avaliadas antes da definição das variáveis que permanecerão na camada Silver.

In [47]:
df_aeps_historico_contrib_silver = (
    df_aeps_historico_contrib
    .iloc[6:27]
    .copy()
)

df_aeps_historico_contrib_silver

,0,1,2,3,4,5,6,7,8,9,10
6,2003,39850452,30537383,8395885,4704900,2120751,1570234,917184,254569,580788,81827
7,2004,42084323,32200411,8766902,4592168,3850082,324652,1117010,256764,835901,24345
8,2005,45035035,34687001,9099704,4693028,4114667,292009,1248330,280717,943844,23769
9,2006,46676737,36158570,9262079,4725699,4275536,260844,1256088,277327,958280,20481
10,2007,49936338,38788600,9710280,4966598,4484207,259475,1437458,264257,1150856,22345
11,2008,53964928,42076251,10235457,5274811,4699830,260816,1653220,323299,1304610,25311
12,2009,55877835,43439321,10684737,5635715,4785919,263103,1753777,359217,1368463,26097
13,2010,60197924,46683012,11548708,6360555,4911110,277043,1966204,460542,1474535,31127
14,2011,64292255,49690560,12441787,7182932,4975675,283180,2159908,569239,1555705,34964
15,2012,67246063,51609519,13333407,8023899,5017088,292420,2303137,674815,1589045,39277


## Seleção das variáveis relevantes

Após a seleção dos registros anuais válidos, será realizada a definição das variáveis que permanecerão na série histórica de contribuintes da camada Silver.

Para o objetivo do projeto, será preservada a quantidade total de contribuintes pessoas físicas e sua composição entre trabalhadores empregados, outros contribuintes e pessoas registradas em ambas as categorias.

Essa estrutura permite acompanhar não apenas a evolução do número total de contribuintes entre 2003 e 2023, mas também possíveis alterações na composição da base contributiva ao longo do período.

As demais colunas da tabela original apresentam detalhamentos das formas de contribuição e serão desconsideradas nesta etapa por não serem necessárias para as análises centrais propostas no projeto.

In [48]:
df_aeps_historico_contrib_silver = (
    df_aeps_historico_contrib_silver[
        [0, 1, 2, 3, 7]
    ]
    .copy()
)

df_aeps_historico_contrib_silver.columns = [
    "ano",
    "total_contribuintes",
    "trabalhadores_empregados",
    "outros_contribuintes",
    "ambos"
]

df_aeps_historico_contrib_silver

,ano,total_contribuintes,trabalhadores_empregados,outros_contribuintes,ambos
6,2003,39850452,30537383,8395885,917184
7,2004,42084323,32200411,8766902,1117010
8,2005,45035035,34687001,9099704,1248330
9,2006,46676737,36158570,9262079,1256088
10,2007,49936338,38788600,9710280,1437458
11,2008,53964928,42076251,10235457,1653220
12,2009,55877835,43439321,10684737,1753777
13,2010,60197924,46683012,11548708,1966204
14,2011,64292255,49690560,12441787,2159908
15,2012,67246063,51609519,13333407,2303137


In [49]:
df_aeps_historico_contrib_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 6 to 26
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   ano                       21 non-null     object
 1   total_contribuintes       21 non-null     object
 2   trabalhadores_empregados  21 non-null     object
 3   outros_contribuintes      21 non-null     object
 4   ambos                     21 non-null     object
dtypes: object(5)
memory usage: 972.0+ bytes


## Conversão dos tipos de dados

A verificação da estrutura da série histórica mostrou que todas as variáveis foram interpretadas inicialmente como `object`, consequência da preservação da estrutura original do arquivo durante a camada Bronze.

Como os registros selecionados representam o ano e quantidades de contribuintes, as cinco variáveis serão convertidas para valores inteiros, adequando seus tipos ao significado dos dados e permitindo as validações e análises posteriores.

In [51]:
colunas_numericas = [
    "ano",
    "total_contribuintes",
    "trabalhadores_empregados",
    "outros_contribuintes",
    "ambos"
]

df_aeps_historico_contrib_silver[colunas_numericas] = (
    df_aeps_historico_contrib_silver[colunas_numericas]
    .astype("int64")
)

df_aeps_historico_contrib_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 6 to 26
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   ano                       21 non-null     int64
 1   total_contribuintes       21 non-null     int64
 2   trabalhadores_empregados  21 non-null     int64
 3   outros_contribuintes      21 non-null     int64
 4   ambos                     21 non-null     int64
dtypes: int64(5)
memory usage: 972.0 bytes


## Verificação de valores ausentes

Após a conversão dos tipos de dados, será verificada a existência de valores ausentes na série histórica.

Essa validação permite confirmar se todos os anos e as respectivas quantidades de contribuintes possuem informações disponíveis antes das verificações de consistência dos dados.

In [52]:
df_aeps_historico_contrib_silver.isnull().sum()

ano                         0
total_contribuintes         0
trabalhadores_empregados    0
outros_contribuintes        0
ambos                       0
dtype: int64

## Verificação de registros duplicados

Com a ausência de valores nulos confirmada, será verificada a existência de registros duplicados na série histórica.

Além da duplicidade completa das linhas, será analisada especificamente a variável `ano`, pois a estrutura esperada para essa base é de uma única observação para cada ano entre 2003 e 2023.

In [53]:
print(
    "Registros duplicados:",
    df_aeps_historico_contrib_silver.duplicated().sum()
)

print(
    "Anos duplicados:",
    df_aeps_historico_contrib_silver.duplicated(
        subset=["ano"]
    ).sum()
)

Registros duplicados: 0
Anos duplicados: 0


## Validação da consistência dos totais

Após as verificações estruturais, será analisada a relação entre o total de contribuintes e as categorias presentes na série histórica.

A tabela do AEPS apresenta separadamente os trabalhadores empregados, os outros contribuintes e os indivíduos contabilizados em ambas as categorias. Como uma mesma pessoa pode estar presente simultaneamente nos dois grupos, a coluna `ambos` deve ser considerada para evitar dupla contagem.

Nesta etapa será verificado se a relação entre essas variáveis reproduz o total de contribuintes informado pela fonte.

In [54]:
df_aeps_historico_contrib_silver["total_calculado"] = (
    df_aeps_historico_contrib_silver["trabalhadores_empregados"]
    + df_aeps_historico_contrib_silver["outros_contribuintes"]
    - df_aeps_historico_contrib_silver["ambos"]
)

df_aeps_historico_contrib_silver["diferenca_total"] = (
    df_aeps_historico_contrib_silver["total_contribuintes"]
    - df_aeps_historico_contrib_silver["total_calculado"]
)

df_aeps_historico_contrib_silver[
    [
        "ano",
        "total_contribuintes",
        "total_calculado",
        "diferenca_total"
    ]
]

,ano,total_contribuintes,total_calculado,diferenca_total
6,2003,39850452,38016084,1834368
7,2004,42084323,39850303,2234020
8,2005,45035035,42538375,2496660
9,2006,46676737,44164561,2512176
10,2007,49936338,47061422,2874916
11,2008,53964928,50658488,3306440
12,2009,55877835,52370281,3507554
13,2010,60197924,56265516,3932408
14,2011,64292255,59972439,4319816
15,2012,67246063,62639789,4606274


## Validação final da camada Silver

Com o tratamento individual das três fontes concluído, será realizada uma verificação final dos arquivos armazenados na camada Silver.

O objetivo é confirmar que todas as bases tratadas foram geradas corretamente e estão disponíveis para as próximas etapas da Arquitetura Medalhão.

In [42]:
arquivos_silver = list(pasta_silver.glob("*.csv"))

for arquivo in arquivos_silver:
    print(arquivo.name)

aeps_contribuintes_tratado.csv
ibge_grupos_etarios_tratado.csv
ibge_indicadores_tratado.csv


### Verificação dos arquivos tratados

Após confirmar a existência dos três arquivos da camada Silver, será realizada uma nova leitura das bases tratadas.

Essa verificação permite confirmar que os arquivos foram armazenados corretamente e que suas estruturas permanecem disponíveis para utilização nas próximas etapas do projeto.

In [43]:
teste_tab3_silver = pd.read_csv(pasta_silver / "ibge_grupos_etarios_tratado.csv")
teste_tab4_silver = pd.read_csv(pasta_silver / "ibge_indicadores_tratado.csv")
teste_aeps_silver = pd.read_csv(pasta_silver / "aeps_contribuintes_tratado.csv")

print("IBGE - Grupos etários:", teste_tab3_silver.shape)
print("IBGE - Indicadores:", teste_tab4_silver.shape)
print("AEPS - Contribuintes:", teste_aeps_silver.shape)

IBGE - Grupos etários: (2343, 15)
IBGE - Indicadores: (2343, 15)
AEPS - Contribuintes: (42, 6)


### Conclusão da camada Silver

A validação final confirmou que os três arquivos tratados foram armazenados corretamente e mantiveram as estruturas definidas durante o processo de tratamento.

A camada Silver está, portanto, concluída, contendo dados padronizados, validados e preparados para integração e análise.

A próxima etapa será a construção da camada Gold, na qual as informações das diferentes fontes serão relacionadas para geração dos indicadores e análises do estudo demográfico e previdenciário.